In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import FloatSlider, IntSlider, HTML, HBox, VBox, Layout
from IPython.display import display

# ============================================================
# GENERAL POLE-ZERO IIR LATTICE-LADDER — SINGLE CANVAS
# ============================================================

plt.ioff()

CONTENT_WIDTH = '900px'
MAX_STAGES = 4
N = 100

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12.5,'axes.labelsize':10.5,'xtick.labelsize':9.5,'ytick.labelsize':9.5,'legend.fontsize':8.8})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.ll-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.ll-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:10px 14px;
    border-radius:8px 8px 0 0;
    font-size:19px;
    font-weight:bold;
}

.ll-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:9px 13px;
    border-radius:0 0 8px 8px;
    font-size:14px;
    line-height:1.48;
    margin-bottom:7px;
}

.ll-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:8px 11px;
    margin-bottom:6px;
    font-size:13.5px;
    line-height:1.45;
}

.ll-result{
    background:#fff8e6;
    border:1px solid #d8b451;
}

.ll-title{
    color:#0d47a1;
    font-weight:bold;
    font-size:14.5px;
    margin-bottom:5px;
}

.ll-equation{
    text-align:center;
    font-family:serif;
    font-size:16px;
    margin:6px 0;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="ll-root">

<div class="ll-header">
General Pole-Zero IIR Lattice-Ladder Implementation
</div>

<div class="ll-doc">

The general IIR lattice-ladder structure extends the all-pole lattice by
forming the final output as a weighted sum of the internal backward
signals g<sub>m</sub>[n].

The denominator is determined by the reflection coefficients
K₁, K₂, ..., K<sub>N</sub>:

<div class="ll-equation">
<b>
A<sub>N</sub>(z)
=
1 + Σ α<sub>N</sub>[k]z<sup>−k</sup>.
</b>
</div>

The final output is

<div class="ll-equation">
<b>
y[n]
=
Σ<sub>m=0</sub><sup>M</sup>
v<sub>m</sub>g<sub>m</sub>[n].
</b>
</div>

The complete transfer function is

<div class="ll-equation">
<b>
H(z)
=
C<sub>M</sub>(z) / A<sub>N</sub>(z),
</b>
</div>

where

<div class="ll-equation">
<b>
C<sub>M</sub>(z)
=
Σ<sub>m=0</sub><sup>M</sup>
v<sub>m</sub>B<sub>m</sub>(z).
</b>
</div>

<div class="ll-equation">
<b>
reflection coefficients K<sub>m</sub> → poles
&nbsp;&nbsp;&nbsp;&nbsp;
ladder coefficients v<sub>m</sub> → zeros
</b>
</div>

Only coefficients belonging to the currently active stages are enabled.

</div>

</div>
"""))

# ============================================================
# CONTROLS
# ============================================================

stage_slider = IntSlider(value=4,min=1,max=MAX_STAGES,step=1,description='Stages:',continuous_update=True,style={'description_width':'45px'},layout=Layout(width='170px'))

K1_slider = FloatSlider(value=0.35,min=-0.85,max=0.85,step=0.05,description='K₁:',continuous_update=True,readout_format='.2f',style={'description_width':'28px'},layout=Layout(width='175px'))

K2_slider = FloatSlider(value=-0.30,min=-0.85,max=0.85,step=0.05,description='K₂:',continuous_update=True,readout_format='.2f',style={'description_width':'28px'},layout=Layout(width='175px'))

K3_slider = FloatSlider(value=0.25,min=-0.85,max=0.85,step=0.05,description='K₃:',continuous_update=True,readout_format='.2f',style={'description_width':'28px'},layout=Layout(width='175px'))

K4_slider = FloatSlider(value=-0.20,min=-0.85,max=0.85,step=0.05,description='K₄:',continuous_update=True,readout_format='.2f',style={'description_width':'28px'},layout=Layout(width='175px'))

v0_slider = FloatSlider(value=0.80,min=-1.00,max=1.00,step=0.05,description='v₀:',continuous_update=True,readout_format='.2f',style={'description_width':'28px'},layout=Layout(width='175px'))

v1_slider = FloatSlider(value=-0.40,min=-1.00,max=1.00,step=0.05,description='v₁:',continuous_update=True,readout_format='.2f',style={'description_width':'28px'},layout=Layout(width='175px'))

v2_slider = FloatSlider(value=0.25,min=-1.00,max=1.00,step=0.05,description='v₂:',continuous_update=True,readout_format='.2f',style={'description_width':'28px'},layout=Layout(width='175px'))

v3_slider = FloatSlider(value=0.15,min=-1.00,max=1.00,step=0.05,description='v₃:',continuous_update=True,readout_format='.2f',style={'description_width':'28px'},layout=Layout(width='175px'))

v4_slider = FloatSlider(value=-0.10,min=-1.00,max=1.00,step=0.05,description='v₄:',continuous_update=True,readout_format='.2f',style={'description_width':'28px'},layout=Layout(width='175px'))

K_sliders = [K1_slider,K2_slider,K3_slider,K4_slider]

v_sliders = [v0_slider,v1_slider,v2_slider,v3_slider,v4_slider]

controls_row1 = HBox([stage_slider,K1_slider,K2_slider,K3_slider,K4_slider],layout=Layout(width=CONTENT_WIDTH))

controls_row2 = HBox([v0_slider,v1_slider,v2_slider,v3_slider,v4_slider],layout=Layout(width=CONTENT_WIDTH))

controls = VBox([controls_row1,controls_row2],layout=Layout(width=CONTENT_WIDTH,border='1px solid #b9cce5',padding='6px 8px',margin='0 0 5px 0'))

# ============================================================
# TEST INPUT
# ============================================================

n = np.arange(N)

x = np.zeros(N)

x[0] = 1.0

x += 0.22*np.sin(0.18*np.pi*n)

x += 0.12*np.sin(0.55*np.pi*n)

# ============================================================
# COEFFICIENT FUNCTIONS
# ============================================================

def get_reflection_coefficients():

    return np.array([K1_slider.value,K2_slider.value,K3_slider.value,K4_slider.value])

def get_ladder_coefficients():

    return np.array([v0_slider.value,v1_slider.value,v2_slider.value,v3_slider.value,v4_slider.value])

def reflection_to_polynomials(K):

    A = np.array([1.0])
    B = np.array([1.0])

    A_history = [A.copy()]
    B_history = [B.copy()]

    for Km in K:

        A_pad = np.append(A,0.0)

        B_shift = np.insert(B,0,0.0)

        A_new = A_pad+Km*B_shift

        B_new = Km*A_pad+B_shift

        A = A_new
        B = B_new

        A_history.append(A.copy())

        B_history.append(B.copy())

    return A,B,A_history,B_history

def ladder_to_numerator(v,B_history,order):

    C = np.zeros(order+1)

    for m in range(order+1):

        C[:m+1] += v[m]*B_history[m]

    return C

# ============================================================
# LATTICE STATES
# ============================================================

def iir_lattice_states(x,K):

    M = len(K)

    f_history = np.zeros((M+1,len(x)))

    g_history = np.zeros((M+1,len(x)))

    previous_g = np.zeros(M+1)

    for sample in range(len(x)):

        f_current = np.zeros(M+1)

        g_current = np.zeros(M+1)

        f_current[M] = x[sample]

        for m in range(M,0,-1):

            f_current[m-1] = f_current[m]-K[m-1]*previous_g[m-1]

        g_current[0] = f_current[0]

        for m in range(1,M+1):

            g_current[m] = K[m-1]*f_current[m-1]+previous_g[m-1]

        f_history[:,sample] = f_current

        g_history[:,sample] = g_current

        previous_g = g_current.copy()

    return f_history,g_history

def lattice_ladder_output(x,K,v):

    f_history,g_history = iir_lattice_states(x,K)

    y = np.zeros(len(x))

    contributions = []

    for m in range(len(v)):

        contribution = v[m]*g_history[m]

        contributions.append(contribution)

        y += contribution

    return y,f_history,g_history,contributions

# ============================================================
# PRECOMPUTE TIME-DOMAIN LIMITS
# ============================================================

test_K_values = [-0.85,0.85]

test_v_values = [-1.0,1.0]

all_internal_values = []

all_contribution_values = []

all_output_values = []

for K1 in test_K_values:

    for K2 in test_K_values:

        for K3 in test_K_values:

            for K4 in test_K_values:

                K_test = np.array([K1,K2,K3,K4])

                A_test,B_test,A_hist_test,B_hist_test = reflection_to_polynomials(K_test)

                for v_sign in test_v_values:

                    v_test = np.array([v_sign,v_sign,v_sign,v_sign,v_sign])

                    C_test = ladder_to_numerator(v_test,B_hist_test,MAX_STAGES)

                    y_lattice_test,f_hist_test,g_hist_test,contrib_test = lattice_ladder_output(x,K_test,v_test)

                    y_direct_test = signal.lfilter(C_test,A_test,x)

                    for m in range(MAX_STAGES+1):

                        all_internal_values.extend(g_hist_test[m])

                        all_contribution_values.extend(contrib_test[m])

                    all_output_values.extend(y_lattice_test)

                    all_output_values.extend(y_direct_test)

internal_limit = 1.10*max(np.max(np.abs(all_internal_values)),1.0)

contribution_limit = 1.10*max(np.max(np.abs(all_contribution_values)),1.0)

output_limit = 1.10*max(np.max(np.abs(all_output_values)),1.0)

# ============================================================
# INITIAL VALUES
# ============================================================

active_stages = stage_slider.value

K_all = get_reflection_coefficients()

v_all = get_ladder_coefficients()

K = K_all[:active_stages]

v = v_all[:active_stages+1]

A,B,A_history,B_history = reflection_to_polynomials(K)

C = ladder_to_numerator(v,B_history,active_stages)

y_lattice,f_history,g_history,contributions = lattice_ladder_output(x,K,v)

y_direct = signal.lfilter(C,A,x)

difference = y_direct-y_lattice

omega,H_direct = signal.freqz(C,A,worN=1024)

poles = np.roots(A)

zeros = np.roots(C) if len(C) > 1 else np.array([])

# ============================================================
# RESULT BOX
# ============================================================

result_html = HTML(layout=Layout(width=CONTENT_WIDTH))

# ============================================================
# SINGLE FIGURE / SINGLE CANVAS
# ============================================================

fig = plt.figure(figsize=(9.0,14.0))

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False

gs = fig.add_gridspec(5,2,height_ratios=[1.15,1.00,1.05,1.00,1.00],hspace=0.55,wspace=0.30)

ax_structure = fig.add_subplot(gs[0,:])

ax_denominator = fig.add_subplot(gs[1,0])

ax_numerator = fig.add_subplot(gs[1,1])

ax_response = fig.add_subplot(gs[2,0])

ax_pz = fig.add_subplot(gs[2,1])

ax_gsignals = fig.add_subplot(gs[3,0])

ax_contributions = fig.add_subplot(gs[3,1])

ax_output = fig.add_subplot(gs[4,0])

ax_difference = fig.add_subplot(gs[4,1])

# ============================================================
# STRUCTURE
# ============================================================

ax_structure.set_xlim(0,11)

ax_structure.set_ylim(-2.4,2.0)

ax_structure.axis('off')

ax_structure.set_title('General Pole-Zero IIR Lattice-Ladder Structure')

stage_centers = [2.4,4.5,6.6,8.7]

stage_rectangles = []

stage_labels = []

K_labels = []

v_labels = []

ax_structure.text(10.35,1.02,r'$f_N[n]=x[n]$',fontsize=10.5,fontweight='bold',ha='center')

ax_structure.text(0.55,1.02,r'$f_0[n]$',fontsize=10.5,fontweight='bold',ha='center')

ax_structure.text(0.55,-0.72,r'$g_0[n]$',fontsize=10.5,fontweight='bold',ha='center')

for m,xc in enumerate(stage_centers):

    rect = plt.Rectangle((xc-0.72,-1.10),1.44,2.35,fill=False,linewidth=1.3)

    ax_structure.add_patch(rect)

    stage_rectangles.append(rect)

    label = ax_structure.text(xc,1.48,f'Stage {m+1}',ha='center',fontsize=10.2,fontweight='bold')

    stage_labels.append(label)

    K_label = ax_structure.text(xc,0.12,'',ha='center',va='center',fontsize=9.8)

    K_labels.append(K_label)

    ax_structure.annotate('',xy=(xc-0.55,0.72),xytext=(xc+0.55,0.72),arrowprops={'arrowstyle':'->','linewidth':1.2})

    ax_structure.annotate('',xy=(xc+0.55,-0.62),xytext=(xc-0.55,-0.62),arrowprops={'arrowstyle':'->','linewidth':1.2})

    ax_structure.annotate('',xy=(xc-0.52,0.70),xytext=(xc-0.52,-0.60),arrowprops={'arrowstyle':'->','linewidth':1.0})

    ax_structure.annotate('',xy=(xc+0.52,-0.60),xytext=(xc-0.48,0.70),arrowprops={'arrowstyle':'->','linewidth':1.0})

    ax_structure.text(xc-0.18,-0.96,r'$z^{-1}$',ha='center',fontsize=9.2)

for m in range(MAX_STAGES-1):

    left_x = stage_centers[m]

    right_x = stage_centers[m+1]

    ax_structure.annotate('',xy=(left_x+0.72,0.72),xytext=(right_x-0.72,0.72),arrowprops={'arrowstyle':'->','linewidth':1.2})

    ax_structure.annotate('',xy=(right_x-0.72,-0.62),xytext=(left_x+0.72,-0.62),arrowprops={'arrowstyle':'->','linewidth':1.2})

ax_structure.annotate('',xy=(9.42,0.72),xytext=(10.15,0.72),arrowprops={'arrowstyle':'->','linewidth':1.3})

ax_structure.annotate('',xy=(1.68,0.72),xytext=(0.90,0.72),arrowprops={'arrowstyle':'->','linewidth':1.3})

ax_structure.annotate('',xy=(1.68,-0.62),xytext=(0.90,-0.62),arrowprops={'arrowstyle':'->','linewidth':1.3})

tap_x_positions = [0.90,2.95,5.05,7.15,9.25]

for m,tap_x in enumerate(tap_x_positions):

    ax_structure.annotate('',xy=(tap_x,-1.55),xytext=(tap_x,-0.64),arrowprops={'arrowstyle':'->','linewidth':1.0})

    label = ax_structure.text(tap_x,-1.78,'',ha='center',fontsize=9.5)

    v_labels.append(label)

ax_structure.plot([0.90,9.55],[-2.05,-2.05],linewidth=1.2)

sum_circle = plt.Circle((9.85,-2.05),0.20,fill=False,linewidth=1.3)

ax_structure.add_patch(sum_circle)

ax_structure.text(9.85,-2.05,r'$\Sigma$',ha='center',va='center',fontsize=11,fontweight='bold')

ax_structure.annotate('',xy=(10.65,-2.05),xytext=(10.05,-2.05),arrowprops={'arrowstyle':'->','linewidth':1.3})

ax_structure.text(10.78,-1.86,r'$y[n]$',fontsize=10.5,fontweight='bold')

# ============================================================
# COEFFICIENT PLOTS
# ============================================================

coefficient_indices = np.arange(MAX_STAGES+1)

initial_A = np.zeros(MAX_STAGES+1)

initial_A[:len(A)] = A

denominator_stems = ax_denominator.vlines(coefficient_indices,0,initial_A,linewidth=1.4)

denominator_markers, = ax_denominator.plot(coefficient_indices,initial_A,'o',markersize=5)

ax_denominator.axhline(0,linewidth=0.8)

ax_denominator.set_xlim(-0.5,MAX_STAGES+0.5)

ax_denominator.set_ylim(-2.5,2.5)

ax_denominator.set_xticks(coefficient_indices)

ax_denominator.set_title('Equivalent Direct-Form Denominator')

ax_denominator.set_xlabel('Coefficient index k')

ax_denominator.set_ylabel(r'$\alpha_N[k]$')

ax_denominator.grid(True,linestyle=':',alpha=0.25)

initial_C = np.zeros(MAX_STAGES+1)

initial_C[:len(C)] = C

numerator_stems = ax_numerator.vlines(coefficient_indices,0,initial_C,linewidth=1.4)

numerator_markers, = ax_numerator.plot(coefficient_indices,initial_C,'o',markersize=5)

ax_numerator.axhline(0,linewidth=0.8)

ax_numerator.set_xlim(-0.5,MAX_STAGES+0.5)

ax_numerator.set_ylim(-3.5,3.5)

ax_numerator.set_xticks(coefficient_indices)

ax_numerator.set_title('Equivalent Direct-Form Numerator')

ax_numerator.set_xlabel('Coefficient index k')

ax_numerator.set_ylabel(r'$\gamma_M[k]$')

ax_numerator.grid(True,linestyle=':',alpha=0.25)

# ============================================================
# FREQUENCY RESPONSE
# ============================================================

response_line, = ax_response.plot(omega/np.pi,np.abs(H_direct),linewidth=1.4)

ax_response.set_xlim(0,1)

ax_response.set_ylim(0,max(1.0,1.10*np.max(np.abs(H_direct))))

ax_response.set_title('General IIR Frequency Response')

ax_response.set_xlabel(r'Normalized frequency $\omega/\pi$')

ax_response.set_ylabel(r'$|H(e^{j\omega})|$')

ax_response.grid(True,linestyle=':',alpha=0.25)

# ============================================================
# POLE-ZERO MAP
# ============================================================

theta = np.linspace(0,2*np.pi,400)

ax_pz.plot(np.cos(theta),np.sin(theta),'--',linewidth=0.9)

ax_pz.axhline(0,linewidth=0.8)

ax_pz.axvline(0,linewidth=0.8)

pole_markers, = ax_pz.plot(np.real(poles),np.imag(poles),'x',markersize=8,markeredgewidth=1.6,label='Poles')

zero_markers, = ax_pz.plot(np.real(zeros),np.imag(zeros),'o',fillstyle='none',markersize=7,markeredgewidth=1.3,label='Zeros')

ax_pz.set_xlim(-2.0,2.0)

ax_pz.set_ylim(-2.0,2.0)

ax_pz.set_aspect('equal',adjustable='box')

ax_pz.set_title('Pole-Zero Locations')

ax_pz.set_xlabel('Real')

ax_pz.set_ylabel('Imaginary')

ax_pz.grid(True,linestyle=':',alpha=0.25)

ax_pz.legend(loc='upper right')

# ============================================================
# INTERNAL SIGNALS
# ============================================================

g_lines = []

for m in range(MAX_STAGES+1):

    data = g_history[m] if m <= active_stages else np.full(N,np.nan)

    line, = ax_gsignals.plot(n,data,linewidth=1.05,label=rf'$g_{m}[n]$')

    g_lines.append(line)

ax_gsignals.set_xlim(0,N-1)

ax_gsignals.set_ylim(-internal_limit,internal_limit)

ax_gsignals.set_title('Internal Backward Signals')

ax_gsignals.set_xlabel('Sample index n')

ax_gsignals.set_ylabel(r'$g_m[n]$')

ax_gsignals.grid(True,linestyle=':',alpha=0.30)

ax_gsignals.legend(loc='upper right',ncol=2)

contribution_lines = []

for m in range(MAX_STAGES+1):

    data = contributions[m] if m <= active_stages else np.full(N,np.nan)

    line, = ax_contributions.plot(n,data,linewidth=1.05,label=rf'$v_{m}g_{m}[n]$')

    contribution_lines.append(line)

ax_contributions.set_xlim(0,N-1)

ax_contributions.set_ylim(-contribution_limit,contribution_limit)

ax_contributions.set_title('Ladder Output Contributions')

ax_contributions.set_xlabel('Sample index n')

ax_contributions.set_ylabel(r'$v_mg_m[n]$')

ax_contributions.grid(True,linestyle=':',alpha=0.30)

ax_contributions.legend(loc='upper right',ncol=2)

# ============================================================
# OUTPUT
# ============================================================

direct_line, = ax_output.plot(n,y_direct,linewidth=1.5,label='Direct-form IIR')

lattice_line, = ax_output.plot(n,y_lattice,'--',linewidth=1.3,label='Lattice-ladder IIR')

ax_output.set_xlim(0,N-1)

ax_output.set_ylim(-output_limit,output_limit)

ax_output.set_title('Final Output Comparison')

ax_output.set_xlabel('Sample index n')

ax_output.set_ylabel('y[n]')

ax_output.grid(True,linestyle=':',alpha=0.30)

ax_output.legend(loc='upper right')

difference_line, = ax_difference.plot(n,difference,linewidth=1.2)

ax_difference.axhline(0,linewidth=0.8)

ax_difference.set_xlim(0,N-1)

ax_difference.set_ylim(-1e-12,1e-12)

ax_difference.set_title('Numerical Difference')

ax_difference.set_xlabel('Sample index n')

ax_difference.set_ylabel(r'$y_D[n]-y_{LL}[n]$')

ax_difference.grid(True,linestyle=':',alpha=0.30)

# ============================================================
# UPDATE
# ============================================================

def update(change=None):

    active_stages = stage_slider.value

    for i,slider in enumerate(K_sliders):

        slider.disabled = i >= active_stages

    v0_slider.disabled = False

    for i in range(1,MAX_STAGES+1):

        v_sliders[i].disabled = i > active_stages

    K_all = get_reflection_coefficients()

    v_all = get_ladder_coefficients()

    K = K_all[:active_stages]

    v = v_all[:active_stages+1]

    A,B,A_history,B_history = reflection_to_polynomials(K)

    C = ladder_to_numerator(v,B_history,active_stages)

    y_lattice,f_history,g_history,contributions = lattice_ladder_output(x,K,v)

    y_direct = signal.lfilter(C,A,x)

    difference = y_direct-y_lattice

    for m in range(MAX_STAGES):

        K_labels[m].set_text(rf'$K_{m+1}={K_all[m]:.2f}$')

        alpha = 1.0 if m < active_stages else 0.18

        stage_rectangles[m].set_alpha(alpha)

        stage_labels[m].set_alpha(alpha)

        K_labels[m].set_alpha(alpha)

    for m in range(MAX_STAGES+1):

        v_labels[m].set_text(rf'$v_{m}={v_all[m]:.2f}$')

        v_labels[m].set_alpha(1.0 if m <= active_stages else 0.18)

    denominator_values = np.zeros(MAX_STAGES+1)

    numerator_values = np.zeros(MAX_STAGES+1)

    denominator_values[:len(A)] = A

    numerator_values[:len(C)] = C

    denominator_stems.set_segments([[(k,0),(k,denominator_values[k])] for k in range(MAX_STAGES+1)])

    denominator_markers.set_ydata(denominator_values)

    numerator_stems.set_segments([[(k,0),(k,numerator_values[k])] for k in range(MAX_STAGES+1)])

    numerator_markers.set_ydata(numerator_values)

    omega,H = signal.freqz(C,A,worN=1024)

    response_line.set_ydata(np.abs(H))

    ax_response.set_ylim(0,max(1.0,1.10*np.max(np.abs(H))))

    poles = np.roots(A)

    zeros = np.roots(C) if len(C) > 1 else np.array([])

    pole_markers.set_data(np.real(poles),np.imag(poles))

    zero_markers.set_data(np.real(zeros),np.imag(zeros))

    for m in range(MAX_STAGES+1):

        if m <= active_stages:

            g_lines[m].set_ydata(g_history[m])

            contribution_lines[m].set_ydata(contributions[m])

        else:

            g_lines[m].set_ydata(np.full(N,np.nan))

            contribution_lines[m].set_ydata(np.full(N,np.nan))

    direct_line.set_ydata(y_direct)

    lattice_line.set_ydata(y_lattice)

    difference_line.set_ydata(difference)

    maximum_difference = np.max(np.abs(difference))

    maximum_pole_radius = np.max(np.abs(poles))

    K_text = ', '.join([f'K{i+1} = {K[i]:.2f}' for i in range(active_stages)])

    v_text = ', '.join([f'v{i} = {v[i]:.2f}' for i in range(active_stages+1)])

    A_text = ', '.join([f'{value:.6f}' for value in A])

    C_text = ', '.join([f'{value:.6f}' for value in C])

    result_html.value = f"""
    <div class="ll-root">

    <div class="ll-box ll-result">

    <div class="ll-title">
    Current general pole-zero IIR lattice-ladder implementation
    </div>

    <b>Active stages:</b> {active_stages}

    <br><br>

    <b>Reflection coefficients:</b> {K_text}

    <br>

    <b>Ladder coefficients:</b> {v_text}

    <br><br>

    Equivalent direct-form denominator:

    <div class="ll-equation">
    A(z) = [{A_text}]
    </div>

    Equivalent direct-form numerator:

    <div class="ll-equation">
    C(z) = [{C_text}]
    </div>

    Maximum pole radius:

    <b>{maximum_pole_radius:.6f}</b>

    &nbsp;&nbsp;&nbsp;

    Maximum
    |y<sub>direct</sub>[n] − y<sub>lattice-ladder</sub>[n]|:

    <b>{maximum_difference:.3e}</b>

    </div>

    </div>
    """

    fig.canvas.draw()

# ============================================================
# OBSERVERS
# ============================================================

stage_slider.observe(update,names='value')

K1_slider.observe(update,names='value')

K2_slider.observe(update,names='value')

K3_slider.observe(update,names='value')

K4_slider.observe(update,names='value')

v0_slider.observe(update,names='value')

v1_slider.observe(update,names='value')

v2_slider.observe(update,names='value')

v3_slider.observe(update,names='value')

v4_slider.observe(update,names='value')

# ============================================================
# DISPLAY
# ============================================================

display(result_html)

display(controls)

display(fig.canvas)

update()